# CISPO Loss

**CISPO: Clipped IS-weight Policy Optimization**

来自 MiniMax，用于训练 MiniMax-M1 模型 (arXiv:2506.13585)

## 核心动机

### PPO/GRPO 裁剪的「Token 丢弃」问题

PPO 和 GRPO 使用如下形式的裁剪目标：
$$
\text{clip-update} = \min\left(r_t \hat{A}_t,\ \text{clip}(r_t, 1-\varepsilon, 1+\varepsilon) \hat{A}_t\right)
$$

当 $\hat{A}_t > 0$ 且 $r_t > 1+\varepsilon$ 时：
$$
\min(r_t \hat{A}_t,\ (1+\varepsilon)\hat{A}_t) = (1+\varepsilon)\hat{A}_t
$$
这时 **梯度对 $r_t$ 来源（即 $\log\pi_\theta$）不再传播**（因为 $(1+\varepsilon)\hat{A}_t$ 是常数，与当前策略参数无关）。

**实际影响**：
- 「反思性行为」token（例如
, 
, 
）
- 这类 token 在基础模型中概率极低
- 第一次 on-policy 更新后，$r_t$ 可能非常大（已更新，但 old policy 概率仍很低）
- 后续 off-policy 更新中，这些 token 被彻底「裁掉」，不再贡献梯度
- 结果：反思行为的学习被**中断**，阻碍 Long CoT 的涌现

### CISPO 的解决方案

CISPO 不裁剪 **token 更新量**，而是裁剪 **IS 权重本身**（在 REINFORCE 框架下）：

$$
\mathcal{J}_{\text{CISPO}} = \mathbb{E}\left[\frac{1}{\sum_i|o_i|} \sum_{i,t}
\underbrace{\text{sg}\left(\hat{r}_{i,t}\right)}_{\text{权重（stop grad）}} \cdot
\hat{A}_{i,t} \cdot
\underbrace{\log\pi_\theta(o_{i,t}|q,o_{i,<t})}_{\text{梯度从这里流动}}
\right]
$$

其中裁剪后的 IS 权重：
$$
\hat{r}_{i,t} = \text{clip}\left(r_{i,t}(\theta), 1-\varepsilon_{\text{low}}^{\text{IS}}, 1+\varepsilon_{\text{high}}^{\text{IS}}\right)
$$

**关键区别**：CISPO 使用 $\log\pi_\theta$（而非比率 $r_t$）作为梯度来源，
IS 权重仅作为**常数加权系数**（stop gradient），**所有 token 都贡献梯度**。

## CISPO 目标函数

$$
\mathcal{J}_{\text{CISPO}}(\theta) = \mathbb{E}_{(q,a)\sim\mathcal{D},\{o_i\}_{i=1}^G\sim\pi_{\theta_{\text{old}}}}
\frac{1}{\sum_{i=1}^G|o_i|} \sum_{i=1}^G\sum_{t=1}^{|o_i|}
\text{sg}\left(\hat{r}_{i,t}(\theta)\right) \cdot \hat{A}_{i,t} \cdot \log\pi_\theta(o_{i,t}|q,o_{i,<t})
$$

其中 $\text{sg}[\cdot]$ 为 stop-gradient，裁剪后 IS 权重：
$$
\hat{r}_{i,t} = \text{clip}\left(\frac{\pi_\theta(o_{i,t})}{\pi_{\theta_{\text{old}}}(o_{i,t})}, 1-\varepsilon_{\text{low}}^{\text{IS}}, 1+\varepsilon_{\text{high}}^{\text{IS}}\right)
$$

在 MiniMax 的实验中，$\varepsilon_{\text{low}}^{\text{IS}}$ 设置为很大（等效于不设置下界）。

## 梯度对比

**PPO/GRPO 梯度**（$\hat{A}_t > 0$，$r_t > 1+\varepsilon$ 时）：
$$
\nabla_\theta \text{clip-update} = \mathbf{0} \quad \text{（梯度为零，token 被丢弃）}
$$

**CISPO 梯度**（$\hat{A}_t > 0$，$r_t > 1+\varepsilon_{\text{high}}^{\text{IS}}$ 时）：
$$
\nabla_\theta [\hat{r}_{i,t} \cdot \hat{A}_t \cdot \log\pi_\theta] = (1+\varepsilon_{\text{high}}^{\text{IS}}) \cdot \hat{A}_t \cdot \nabla_\theta \log\pi_\theta \neq \mathbf{0}
$$
**所有 token 仍然贡献梯度**，只是权重被上限限制。

## 统一框架（Token-wise Mask）

CISPO 提出了一个统一的损失公式，可以通过调整 mask $M_{i,t}$ 来表示不同的裁剪策略：

$$
J_{\text{unify}}(\theta) = \frac{1}{\sum|o_i|} \sum_{i,t} \text{sg}(\hat{r}_{i,t}) \cdot \hat{A}_{i,t} \cdot \log\pi_\theta(o_{i,t}) \cdot M_{i,t}
$$

其中 mask $M_{i,t}$ 的定义：
- PPO/GRPO：$M_{i,t} = 0$ 若 token 超出裁剪范围（相当于丢弃 token）
- CISPO：$M_{i,t} = 1$（始终为 1，不丢弃任何 token）

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

## 优势函数（与 GRPO/DAPO 相同）

In [ ]:
def grpo_advantage(rewards):
    """
    组相对优势估计（标准 z-score 归一化）
    A_i = (R_i - mean(R)) / (std(R) + ε)
    """
    epsilon = 1e-5
    return (rewards - rewards.mean()) / (rewards.std() + epsilon)

## 核心对比：PPO/GRPO 裁剪 vs CISPO 裁剪

In [ ]:
# 直觉演示：对比 PPO/GRPO 和 CISPO 在大 IS 比率下的行为
# 故事：一个「反思」token（如 "Wait"），基础模型概率很低，
#        RL 第一次更新大幅提升了概率，导致后续 off-policy 时 r_t 很大

# 模拟参数
epsilon = 0.2          # PPO/GRPO 裁剪阈值
epsilon_high_IS = 0.2  # CISPO IS 权重上界
A = 1.5                # 正优势（好的 token，希望增加概率）

# 不同的 IS 比率（r_t = π_θ / π_θold）
ratios = torch.linspace(0.5, 3.0, 100)
log_pi_theta = torch.tensor(-0.5)  # 当前策略 log 概率（固定一个值用于演示）

# PPO/GRPO: min(r*A, clip(r)*A)
ppo_objective = torch.minimum(ratios * A, torch.clamp(ratios, 1-epsilon, 1+epsilon) * A)
ppo_gradient_magnitude = torch.where(
    (ratios > 1 + epsilon) & (A > 0),
    torch.zeros_like(ratios),  # 超出范围，梯度为 0（token 被丢弃）
    torch.ones_like(ratios) * abs(A)  # 未超出，正常梯度
)

# CISPO: sg[clip(r)] * A * log π_θ — 梯度恒不为零
cispo_weight = torch.clamp(ratios, 1-epsilon_high_IS, 1+epsilon_high_IS)  # 裁剪 IS 权重
cispo_gradient_magnitude = cispo_weight * abs(A)  # 梯度大小始终 > 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：目标函数值
axes[0].plot(ratios, ppo_objective.numpy(), 'r-', label='PPO/GRPO objective', linewidth=2)
axes[0].plot(ratios, cispo_weight.numpy() * A, 'b-', label='CISPO weighted target (×log π)', linewidth=2)
axes[0].axvline(x=1+epsilon, color='orange', linestyle='--', label=f'upper clip (ε={epsilon})')
axes[0].axvline(x=1, color='gray', linestyle=':', label='on-policy (r=1)')
axes[0].set_xlabel('IS Ratio r_t')
axes[0].set_ylabel('Objective Value')
axes[0].set_title('目标函数对比（A > 0）')
axes[0].legend()
axes[0].grid()

# 右图：梯度大小
axes[1].plot(ratios, ppo_gradient_magnitude.numpy(), 'r-', label='PPO/GRPO |gradient|', linewidth=2)
axes[1].plot(ratios, cispo_gradient_magnitude.numpy(), 'b-', label='CISPO |gradient|', linewidth=2)
axes[1].axvline(x=1+epsilon, color='orange', linestyle='--', label=f'upper clip (ε={epsilon})')
axes[1].axvline(x=1, color='gray', linestyle=':', label='on-policy (r=1)')
axes[1].set_xlabel('IS Ratio r_t')
axes[1].set_ylabel('Gradient Magnitude')
axes[1].set_title('梯度大小对比（A > 0时）')
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.show()

print('关键发现：')
print(f'  PPO/GRPO: 当 r > {1+epsilon:.1f} 时，梯度为 0（token 被完全丢弃）')
print(f'  CISPO: 当 r > {1+epsilon_high_IS:.1f} 时，梯度仍为 {(1+epsilon_high_IS)*abs(A):.4f} > 0（token 仍有梯度）')
print()
print('这对「反思行为」token（低概率但重要）至关重要：')
print('PPO/GRPO 会在首次更新后丢弃这些 token，阻碍长链式推理的涌现')
print('CISPO 始终保持这些 token 的梯度，有利于反思行为的持续学习')

## CISPO Loss 核心实现

In [ ]:
def cispo_loss(
    pi_logprob,          # 当前策略 log prob，[bs, seq_len]（梯度从这里流动）
    pi_old_logprob,      # 旧策略 log prob，[bs, seq_len]（不需要梯度）
    rewards,             # 组内奖励，[bs]
    input_len,           # prompt 长度
    epsilon_low=100.0,   # IS 权重下界（MiniMax 实验中设为极大，等效于不限制下界）
    epsilon_high=0.2,    # IS 权重上界
    is_debug=True
):
    """
    CISPO Loss 实现
    
    基于 REINFORCE 框架（而非 PPO 框架）：
        REINFORCE: J = E[sum_t r_t(stop_grad) * A * log π_θ]
        CISPO:     J = E[sum_t clip(r_t, ...)(stop_grad) * A * log π_θ]
    
    与 GRPO/PPO 的根本区别：
    - PPO: 目标是 min(r*A, clip(r)*A)，梯度通过 r = π_θ/π_θold 传播
           当 r 超出裁剪范围，梯度为 0（token 被丢弃）
    - CISPO: 目标是 clip(r)(stop_grad) * A * log π_θ，梯度通过 log π_θ 传播
             IS 权重永远是常数（stop gradient），log π_θ 永远有梯度
    
    实现要点：
    1. 计算 IS 比率 r_t = π_θ(o_t) / π_θold(o_t)（在 log 空间计算）
    2. 裁剪 IS 权重：r_hat = clip(r_t, 1-ε_low, 1+ε_high)
    3. Stop gradient：r_hat.detach() — 不允许梯度通过 IS 权重传播
    4. 最终损失：-sum[r_hat.detach() * A * log_pi_theta] / total_tokens
    """
    bs, seq_len = pi_logprob.shape
    
    # ============================================================
    # Step 1: 构建输出 mask
    # ============================================================
    mask = torch.zeros(bs, seq_len)
    mask[:, input_len:] = 1.0
    total_output_tokens = mask.sum()
    
    # ============================================================
    # Step 2: 计算组相对优势
    # ============================================================
    advantage = grpo_advantage(rewards).unsqueeze(1)  # [bs, 1]
    
    # ============================================================
    # Step 3: 计算 IS 比率 r_t = π_θ / π_θold
    # 使用 log 空间计算更稳定
    # ============================================================
    with torch.no_grad():
        # IS 比率（detach，不传播梯度通过这部分）
        ratio = torch.exp(pi_logprob - pi_old_logprob)  # [bs, seq_len]
        
        # 裁剪 IS 权重
        # 注意：上界限制高 IS 权重，防止极端 off-policy 更新
        # MiniMax 实验中只设置上界（epsilon_low 设很大等于不限制下界）
        ratio_hat = torch.clamp(ratio, 1 - epsilon_low, 1 + epsilon_high)  # [bs, seq_len]
    
    # ratio_hat 已经是 detached 的（在 no_grad 块中计算）
    # 这就是 CISPO 的 stop-gradient 操作
    
    # ============================================================
    # Step 4: CISPO 核心损失计算
    # 与 PPO/GRPO 的根本区别：
    # - GRPO:  -(r_t * A)，梯度通过 r_t = π_θ/π_θold 传播
    #          当 r_t 被裁剪后，梯度变为常数（= 0）
    # - CISPO: -(r_hat * A * log_π_θ)，梯度通过 log_π_θ 传播
    #          r_hat 是常数（detached），log_π_θ 始终有梯度
    #
    # 梯度公式对比（对 θ 求导）：
    # GRPO:  ∂/∂θ [-(r_t * A)] = -A * ∂r_t/∂θ = -A * (π_θ/π_θold) * ∂log π_θ/∂θ
    # CISPO: ∂/∂θ [-(r_hat * A * log π_θ)] = -r_hat * A * ∂log π_θ/∂θ
    # 形式上 CISPO 用 r_hat（裁剪后的常数）替代了 r_t（当前策略的函数）
    # ============================================================
    loss_element = ratio_hat * advantage * pi_logprob  # [bs, seq_len]
    loss = -(1.0 / total_output_tokens) * (loss_element * mask).sum()
    
    if is_debug:
        print(f'[Rewards]         : {rewards.tolist()}')
        print(f'[Advantage]       : {advantage.squeeze().tolist()}')
        ratio_masked = ratio[mask == 1]
        print(f'[IS Ratio range]  : [{ratio_masked.min():.4f}, {ratio_masked.max():.4f}]')
        ratio_hat_masked = ratio_hat[mask == 1]
        n_clipped = (ratio[mask==1] > 1+epsilon_high).sum().item()
        n_total = mask.sum().int().item()
        print(f'[Clipped tokens]  : {n_clipped}/{n_total} ({n_clipped/n_total:.1%}) — 这些在 GRPO 中梯度为 0，但 CISPO 仍有梯度')
        print(f'[Loss]            : {loss.item():.6f}')
    
    return loss

## 验证：所有 token 的梯度是否始终非零

In [ ]:
# 验证 CISPO 的核心特性：所有输出 token 都有梯度
# 对比 GRPO 中某些 token 梯度为 0

torch.manual_seed(42)

def grpo_loss_compare(pi_logprob, pi_old_logprob, rewards, input_len, epsilon=0.2):
    """GRPO Loss（对比用）"""
    bs, seq_len = pi_logprob.shape
    mask = torch.zeros(bs, seq_len)
    mask[:, input_len:] = 1.0
    total = mask.sum()
    ratio = torch.exp(pi_logprob - pi_old_logprob.detach())
    ratio_clip = torch.clamp(ratio, 1-epsilon, 1+epsilon)
    advantage = grpo_advantage(rewards).unsqueeze(1)
    pg = torch.minimum(ratio * advantage, ratio_clip * advantage)
    return -(1.0/total) * (pg * mask).sum()


bs, seq_len = 2, 8
input_len = 3
vocab_size = 16

pi_logits = torch.randn(bs, seq_len, vocab_size, requires_grad=False)
token_ids = torch.randint(0, vocab_size, (bs, seq_len))

pi_old_logprob_fixed = F.log_softmax(torch.randn(bs, seq_len, vocab_size), dim=-1)
pi_old_logprob_fixed = torch.gather(pi_old_logprob_fixed, -1, token_ids.unsqueeze(-1)).squeeze(-1)

# 模拟大幅策略更新后的情景（IS 比率可能很大）
# 当前策略比旧策略高得多（模拟首次 on-policy 更新后的 off-policy 场景）
pi_logits_updated = pi_logits.clone() + 2.0  # 大幅提升某些 token 的 logit
pi_logprob_current = F.log_softmax(pi_logits_updated, dim=-1)
pi_logprob_current = torch.gather(pi_logprob_current, -1, token_ids.unsqueeze(-1)).squeeze(-1)
pi_logprob_current.requires_grad_(True)

rewards = torch.tensor([1.0, 0.0])

# 计算 GRPO 梯度
pi_logprob_grpo = pi_logprob_current.clone().detach().requires_grad_(True)
loss_grpo = grpo_loss_compare(pi_logprob_grpo, pi_old_logprob_fixed.detach(), rewards, input_len)
loss_grpo.backward()
grpo_grad = pi_logprob_grpo.grad

# 计算 CISPO 梯度
pi_logprob_cispo = pi_logprob_current.clone().detach().requires_grad_(True)
loss_cispo = cispo_loss(pi_logprob_cispo, pi_old_logprob_fixed.detach(), rewards, input_len, is_debug=False)
loss_cispo.backward()
cispo_grad = pi_logprob_cispo.grad

mask = torch.zeros(bs, seq_len)
mask[:, input_len:] = 1.0

print('输出 token 的梯度对比（每个 token 的梯度绝对值）：')
print('批次 0（奖励=1，正优势）')
for t in range(input_len, seq_len):
    grpo_g = grpo_grad[0, t].item()
    cispo_g = cispo_grad[0, t].item()
    ratio_val = torch.exp(pi_logprob_current[0, t] - pi_old_logprob_fixed[0, t]).item()
    clipped = '← 被裁剪（GRPO梯度为零）' if abs(grpo_g) < 1e-6 else ''
    print(f'  token t={t}: IS ratio={ratio_val:.3f}, GRPO grad={grpo_g:.4f}, CISPO grad={cispo_g:.4f} {clipped}')

print()
grpo_zero_count = (grpo_grad[0, input_len:].abs() < 1e-6).sum().item()
cispo_zero_count = (cispo_grad[0, input_len:].abs() < 1e-6).sum().item()
print(f'GRPO 梯度为零的输出 token 数: {grpo_zero_count}/{seq_len - input_len}')
print(f'CISPO 梯度为零的输出 token 数: {cispo_zero_count}/{seq_len - input_len}')

## 完整 CISPO Loss 测试

In [ ]:
torch.manual_seed(0)
bs, seq_len, vocab_size = 4, 10, 32
input_len = 3

pi_logits = torch.randn(bs, seq_len, vocab_size)
pi_old_logits = torch.randn(bs, seq_len, vocab_size)

token_ids_full = torch.randint(0, vocab_size, (bs, seq_len))

pi_logprob_test = F.log_softmax(pi_logits, dim=-1)
pi_logprob_test = torch.gather(pi_logprob_test, -1, token_ids_full.unsqueeze(-1)).squeeze(-1)

pi_old_logprob_test = F.log_softmax(pi_old_logits, dim=-1)
pi_old_logprob_test = torch.gather(pi_old_logprob_test, -1, token_ids_full.unsqueeze(-1)).squeeze(-1)

rewards_test = torch.tensor([1.0, 1.0, 0.0, 0.0])

print('='*60)
print('CISPO Loss 完整测试')
print('='*60)
loss = cispo_loss(pi_logprob_test, pi_old_logprob_test.detach(), rewards_test, input_len,
                  epsilon_low=100.0, epsilon_high=0.2)

## 统一框架：GRPO、DAPO、CISPO 梯度对比

In [ ]:
# 可视化三种方法在不同 IS 比率下的等效梯度权重
epsilon = 0.2
epsilon_low, epsilon_high = 0.2, 0.28  # DAPO 参数
A = 1.0  # 正优势

ratios = torch.linspace(0.3, 2.5, 200)

# GRPO: 对 log π_θ 的等效梯度权重
# min(r*A, clip(r)*A) 中，当 r > 1+ε，目标固定为 (1+ε)*A，对 log π_θ 梯度贡献 ≈ 0
# 简化：将 min 的结果对 r 求导，得到等效权重对 r 的分段关系
# GRPO 实际上是用 r 作为梯度路径，等效权重 = r（在未裁剪区间）
grpo_effective_weight = torch.where(
    (ratios > 1 + epsilon) & (A > 0),
    torch.zeros_like(ratios),  # 裁剪区域：梯度为 0
    ratios * A  # 正常区域：权重为 r
)

# DAPO: 与 GRPO 类似，但上界更大
dapo_effective_weight = torch.where(
    (ratios > 1 + epsilon_high) & (A > 0),
    torch.zeros_like(ratios),
    ratios * A
)

# CISPO: 对 log π_θ 的等效梯度权重 = clip(r) * A（始终非零）
cispo_effective_weight = torch.clamp(ratios, 1-epsilon, 1+epsilon) * A

plt.figure(figsize=(12, 5))
plt.plot(ratios, grpo_effective_weight.numpy(), 'r-', label='GRPO (ε=0.2)', linewidth=2)
plt.plot(ratios, dapo_effective_weight.numpy(), 'g--', label='DAPO (ε_high=0.28)', linewidth=2)
plt.plot(ratios, cispo_effective_weight.numpy(), 'b-', label='CISPO (ε_high=0.2)', linewidth=2)
plt.axvline(x=1.0, color='gray', linestyle=':', label='on-policy (r=1)')
plt.xlabel('IS Ratio r_t')
plt.ylabel('Effective Gradient Weight (×∇log π_θ)')
plt.title('三种方法对 log π_θ 的等效梯度权重对比（A>0时）')
plt.legend()
plt.grid()
plt.ylim(-0.1, 2.5)
plt.show()

print('关键对比：')
print('- GRPO: r > 1.2 时等效权重 = 0（token 梯度消失）')
print('- DAPO: r > 1.28 时等效权重 = 0（放宽了上界，DAPO 的 Clip-Higher）')
print('- CISPO: 无论 r 多大，等效权重始终 = min(r, 1.2)（所有 token 有梯度）')

## CISPO vs GRPO vs DAPO 总结

| 特性 | GRPO | DAPO | CISPO |
|------|------|------|-------|
| 裁剪目标 | token 更新量 | token 更新量（解耦阈值）| IS 权重本身 |
| 梯度路径 | 通过 $r_t = \pi_\theta/\pi_{\theta_{\text{old}}}$ | 同 GRPO | 通过 $\log\pi_\theta$（REINFORCE 形式）|
| 大 IS 比率时梯度 | **为零**（token 被丢弃）| 接近零（DAPO 延迟了丢弃）| **非零**（权重截断但仍有梯度）|
| KL 正则 | 有 | 无 | 无 |
| 归一化方式 | 序列均值→批次均值 | Token 总数 | Token 总数 |
| 适合场景 | 一般 RL | Long CoT RL | 混合注意力模型、极端 off-policy |
| 代表模型 | DeepSeek-R1 | DAPO论文 | MiniMax-M1 |

**核心直觉**：
- GRPO 是「**裁剪更新步长**」—— 超出范围的 token 更新量被置零，等价于跳过该 token
- CISPO 是「**裁剪加权系数**」—— 超出范围的 token 权重被截断，但 REINFORCE 梯度始终存在
- 这个区别在有大量 off-policy 更新（多 epoch 或 minibatch 训练）时尤为重要